# Paragon — Colab A100 Quickstart

**Regime-Aware Dynamic Asset Allocation** — Cross-Asset Transformer + HMM regime detector + CVaR optimizer.

**Repo:** https://github.com/Tyler-Pellek/regime-aware-allocation

## Before you start
1. `Runtime` → `Change runtime type` → **A100 GPU**.
2. Get your Kaggle API token from https://www.kaggle.com/settings → *Create New Token* → downloads `kaggle.json`.
3. In the Files panel (left sidebar, folder icon), drag-drop `kaggle.json` into `/content/`.

Then run cells top-to-bottom.

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/Tyler-Pellek/regime-aware-allocation.git /content/paragon || (cd /content/paragon && git pull)
%cd /content/paragon
!pip install -q -r requirements.txt

import torch
print('CUDA:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Place your Kaggle API token

In [ ]:
import os, shutil, pathlib
src = pathlib.Path('/content/kaggle.json')
assert src.exists(), 'Upload kaggle.json to /content/ via the Files panel before running this cell.'
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy(src, os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
!kaggle datasets list -s 'stock-market-dataset' --max-size 1 | head -5

## 3. Download the Kaggle stock-market-dataset (~700 MB unzipped)

In [ ]:
!bash scripts/download_kaggle.sh
!echo 'Stock CSVs:'; ls data/raw/kaggle_stock_market/stocks/ | wc -l
!echo 'ETF CSVs:'; ls data/raw/kaggle_stock_market/etfs/ | wc -l

## 4. (Optional) Smoke test on synthetic data — should finish in ~10 s

In [ ]:
!pip install -q pytest && PYTHONPATH=. pytest -q tests/

## 5. Run the full A100 backtest

Default config: 2005-2020 weekly rebalance, d_model=128, 4 layers, 30 epochs, periodic retrain per fold.

Expected runtime on A100: **~20-45 min** depending on number of folds. Streaming output via `python -u` so fold/epoch progress is visible live.

In [ ]:
!PYTHONPATH=. python -u scripts/run_backtest.py --config configs/default.yaml

## 6. Headline metrics as a clean table

In [ ]:
import json, pathlib, pandas as pd
summary = json.loads(pathlib.Path('artifacts/reports/default/summary.json').read_text())
rows = []
for name, m in summary.items():
    rows.append({
        'strategy': name,
        'CAGR':   f"{m.get('cagr', 0)*100:6.2f}%",
        'Vol':    f"{m.get('ann_vol', 0)*100:6.2f}%",
        'Sharpe': f"{m.get('sharpe', 0):5.2f}",
        'MaxDD':  f"{m.get('max_drawdown', 0)*100:6.2f}%",
        'Calmar': f"{m.get('calmar', 0):5.2f}",
        'Alpha':  f"{m.get('alpha_ann', 0)*100:6.2f}%" if 'alpha_ann' in m else '',
    })
print(pd.DataFrame(rows).to_string(index=False))

## 7. View the report figures inline

In [ ]:
from IPython.display import Image, display
for png in ['equity_curves.png', 'drawdown.png', 'regime_overlay.png', 'weights_heatmap.png', 'rolling_sharpe.png', 'turnover.png']:
    p = f'artifacts/reports/default/figures/{png}'
    print(f'\n--- {png} ---')
    display(Image(filename=p))

## 8. Zip + download the full report

In [ ]:
!cd artifacts/reports && zip -qr /content/paragon_default_report.zip default
from google.colab import files
files.download('/content/paragon_default_report.zip')

## 9. (Optional) Iterate — pull latest changes and re-run

After editing the repo locally and `git push`-ing, run the cell below to pick up changes without re-downloading the Kaggle data. Then re-run Cell 5.

In [ ]:
%cd /content/paragon
!git pull